In [3]:
import yfinance as yf
import pandas as pd

# 1. 銘柄の指定（例として米国株・ETFで超メジャーな「VOO」を使用）
ticker_symbol = "VOO"
ticker = yf.Ticker(ticker_symbol)

print(f"=== 【{ticker_symbol}】のデータ取得を開始します ===")

# ==========================================
# ① 基本情報（メタデータ）の取得
# ==========================================
print("\n--- ① 基本情報 ---")
info = ticker.info

# 辞書型から安全にデータを抽出（キーがない場合の対策で .get() を使用）
print(f"ファンド正式名 : {info.get('longName')}")
print(f"運用会社       : {info.get('fundFamily')}")
# 修正前：print(f"総資産額 (Nav) : ${info.get('totalAssets'):, if info.get('totalAssets') else 0}")

# 修正後：一度変数に安全に取得してから、フォーマットを適用します
assets = info.get('totalAssets')
print(f"総資産額 (Nav) : ${assets:,}" if assets else "総資産額 (Nav) : データなし")
print(f"経費率(信託報酬): {info.get('expenseRatio', 0) * 100:.2f}%")

# ==========================================
# ② 過去の価格データ（株価・基準価額）の取得
# ==========================================
print("\n--- ② 過去の価格履歴（直近5日分を表示） ---")
# period="1y" で過去1年分を取得。機械学習に使うなら "5y" や "max" に変更
history_df = ticker.history(period="1y")

# 直近の5行だけ表示
print(history_df[['Open', 'High', 'Low', 'Close', 'Volume']].tail())

# ==========================================
# ③ 分配金（配当）の履歴
# ==========================================
print("\n--- ③ 直近の分配金履歴（最後の5件を表示） ---")
dividends = ticker.dividends

if not dividends.empty:
    print(dividends.tail())
else:
    print("分配金のデータが見つかりませんでした。")

# ==========================================
# ④ 内部の保有銘柄（トップ10）の情報
# ==========================================
print("\n--- ④ 組入上位10銘柄 ---")
try:
    # 投資信託・ETF専用のデータオブジェクトを取得
    funds_data = ticker.funds_data
    top_holdings = funds_data.top_holdings

    # 見やすくするために pandas の DataFrame として表示
    print(top_holdings)
except AttributeError:
    print("※この銘柄の組入銘柄データは yfinance から取得できませんでした（個別株など）。")
except Exception as e:
    print(f"データ取得中にエラーが発生しました: {e}")

print("\n=== すべてのデータ取得が完了しました！ ===")

=== 【VOO】のデータ取得を開始します ===

--- ① 基本情報 ---
ファンド正式名 : Vanguard S&P 500 ETF
運用会社       : Vanguard
総資産額 (Nav) : $1,701,513,003,008
経費率(信託報酬): 0.00%

--- ② 過去の価格履歴（直近5日分を表示） ---
                                 Open        High         Low       Close  \
Date                                                                        
2026-06-05 00:00:00-04:00  691.710022  692.179993  676.250000  678.000000   
2026-06-08 00:00:00-04:00  683.469971  685.289978  678.700012  679.679993   
2026-06-09 00:00:00-04:00  683.710022  686.760010  664.320007  677.700012   
2026-06-10 00:00:00-04:00  674.320007  678.880005  666.880005  667.049988   
2026-06-11 00:00:00-04:00  670.099976  680.380005  666.000000  678.229980   

                             Volume  
Date                                 
2026-06-05 00:00:00-04:00  10846500  
2026-06-08 00:00:00-04:00   8867200  
2026-06-09 00:00:00-04:00  16197200  
2026-06-10 00:00:00-04:00  16472700  
2026-06-11 00:00:00-04:00  19919900  

--- ③ 直近の分配金履歴（最後の5件

In [4]:
import yfinance as yf

ticker_symbol = "VOO"
ticker = yf.Ticker(ticker_symbol)

# period="10y" で過去10年分を指定
# (土日祝日を除く、市場が動いていた約2500日分のデータが取れます)
history_10y = ticker.history(period="10y")

print(f"=== {ticker_symbol} 過去10年間のデータ ===")
print(f"取得したデータの行数（日数）: {len(history_10y)} 行")

# データの最初（10年前）の5行を表示
print("\n--- データの始まり（10年前） ---")
print(history_10y[['Open', 'High', 'Low', 'Close']].head())

# データの最後（直近）の5行を表示
print("\n--- データの終わり（直近） ---")
print(history_10y[['Open', 'High', 'Low', 'Close']].tail())

=== VOO 過去10年間のデータ ===
取得したデータの行数（日数）: 2514 行

--- データの始まり（10年前） ---
                                 Open        High         Low       Close
Date                                                                     
2016-06-13 00:00:00-04:00  162.442765  163.237935  161.672971  161.757568
2016-06-14 00:00:00-04:00  161.427644  161.952113  160.556344  161.402267
2016-06-15 00:00:00-04:00  161.791381  162.383538  161.030054  161.165405
2016-06-16 00:00:00-04:00  160.387148  161.850593  159.515848  161.689865
2016-06-17 00:00:00-04:00  161.656044  161.656044  160.497120  161.089264

--- データの終わり（直近） ---
                                 Open        High         Low       Close
Date                                                                     
2026-06-05 00:00:00-04:00  691.710022  692.179993  676.250000  678.000000
2026-06-08 00:00:00-04:00  683.469971  685.289978  678.700012  679.679993
2026-06-09 00:00:00-04:00  683.710022  686.760010  664.320007  677.700012
2026-06-10 00:00:00-04

In [5]:
import numpy as np

# 1. 毎日の「変化率（日次リターン）」を計算する
# 前日のClose（終値）から何%動いたかを自動計算してくれます
history_10y['Daily_Return'] = history_10y['Close'].pct_change()

# 2. 【リターン】平均日次リターンを「年率」に換算する
# 1年間の市場の営業日（土日祝を除く）はおおよそ 252日 なので、252倍します
mean_return_daily = history_10y['Daily_Return'].mean()
annual_return = mean_return_daily * 252

# 3. 【リスク】日次リターンの標準偏差を「年率」に換算する
# 統計学のルール上、標準偏差を年率にするには「営業日数のルート（√252）」を掛けます
std_return_daily = history_10y['Daily_Return'].std()
annual_risk = std_return_daily * np.sqrt(252)

# 4. 結果を表示（%表記にするために100を掛けます）
print(f"=== {ticker_symbol} 過去10年間のシミュレーション ===")
print(f"期待リターン（年率）: {annual_return * 100:.2f}%")
print(f"リスク（年率標準偏差）: {annual_risk * 100:.2f}%")

# おまけ：シャープレシオ（効率性の指標。高いほど優秀）
# ※今回は簡易的に無リスク資産の金利を0%として計算
sharp_ratio = annual_return / annual_risk
print(f"シャープレシオ       : {sharp_ratio:.2f}")

=== VOO 過去10年間のシミュレーション ===
期待リターン（年率）: 16.17%
リスク（年率標準偏差）: 18.01%
シャープレシオ       : 0.90
